<a href="https://colab.research.google.com/github/willf123/dbt-analytics-portfolio/blob/main/notebooks/tampa_weather_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install BigQuery client
!pip install google-cloud-bigquery

# Import libraries
from google.colab import auth
from google.cloud import bigquery

# Authenticate with your Google account
auth.authenticate_user()

# Connect to your BigQuery project
project_id = 'crafty-circlet-491120-t3'
client = bigquery.Client(project=project_id)

print("Connected to BigQuery successfully")

Connected to BigQuery successfully


In [2]:
import requests
import json
from datetime import datetime

# Pull weather data for Tampa from Open-Meteo API
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 27.9506,
    "longitude": -82.4572,
    "daily": ["temperature_2m_max", "temperature_2m_min",
               "precipitation_sum", "windspeed_10m_max"],
    "temperature_unit": "fahrenheit",
    "windspeed_unit": "mph",
    "precipitation_unit": "inch",
    "timezone": "America/New_York",
    "forecast_days": 7
}

response = requests.get(url, params=params)
data = response.json()

print(f"Status code: {response.status_code}")
print(f"Days returned: {len(data['daily']['time'])}")
print(json.dumps(data['daily'], indent=2))

Status code: 200
Days returned: 7
{
  "time": [
    "2026-04-22",
    "2026-04-23",
    "2026-04-24",
    "2026-04-25",
    "2026-04-26",
    "2026-04-27",
    "2026-04-28"
  ],
  "temperature_2m_max": [
    81.3,
    85.9,
    86.7,
    85.6,
    86.7,
    87.3,
    87.9
  ],
  "temperature_2m_min": [
    56.2,
    59.1,
    62.2,
    63.7,
    64.6,
    66.0,
    70.3
  ],
  "precipitation_sum": [
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0
  ],
  "windspeed_10m_max": [
    13.1,
    10.7,
    9.2,
    14.0,
    12.0,
    12.7,
    13.8
  ]
}


In [4]:
from google.cloud import bigquery
import pandas as pd

# Parse the API response into a list of rows
daily = data['daily']

rows = []
for i in range(len(daily['time'])):
    rows.append({
        'date': daily['time'][i],
        'temp_max_f': daily['temperature_2m_max'][i],
        'temp_min_f': daily['temperature_2m_min'][i],
        'precipitation_inches': daily['precipitation_sum'][i],
        'windspeed_max_mph': daily['windspeed_10m_max'][i],
        'location': 'Tampa, FL',
        'loaded_at': datetime.utcnow().isoformat()
    })

# Convert to dataframe
df = pd.DataFrame(rows)
print(f"Rows to load: {len(df)}")
print(df)

Rows to load: 7
         date  temp_max_f  temp_min_f  precipitation_inches  \
0  2026-04-22        81.3        56.2                   0.0   
1  2026-04-23        85.9        59.1                   0.0   
2  2026-04-24        86.7        62.2                   0.0   
3  2026-04-25        85.6        63.7                   0.0   
4  2026-04-26        86.7        64.6                   0.0   
5  2026-04-27        87.3        66.0                   0.0   
6  2026-04-28        87.9        70.3                   0.0   

   windspeed_max_mph   location                   loaded_at  
0               13.1  Tampa, FL  2026-04-22T21:10:01.672756  
1               10.7  Tampa, FL  2026-04-22T21:10:01.672781  
2                9.2  Tampa, FL  2026-04-22T21:10:01.672808  
3               14.0  Tampa, FL  2026-04-22T21:10:01.672812  
4               12.0  Tampa, FL  2026-04-22T21:10:01.672816  
5               12.7  Tampa, FL  2026-04-22T21:10:01.672821  
6               13.8  Tampa, FL  2026-04-22T2

/tmp/ipykernel_28247/890358126.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'loaded_at': datetime.utcnow().isoformat()


In [5]:
# Define the destination table
dataset_id = 'dbt_wfiore'
table_id = 'raw_tampa_weather'
full_table_id = f'{project_id}.{dataset_id}.{table_id}'

# Define the schema
schema = [
    bigquery.SchemaField('date', 'STRING'),
    bigquery.SchemaField('temp_max_f', 'FLOAT'),
    bigquery.SchemaField('temp_min_f', 'FLOAT'),
    bigquery.SchemaField('precipitation_inches', 'FLOAT'),
    bigquery.SchemaField('windspeed_max_mph', 'FLOAT'),
    bigquery.SchemaField('location', 'STRING'),
    bigquery.SchemaField('loaded_at', 'STRING'),
]

# Create the table if it doesn't exist
table = bigquery.Table(full_table_id, schema=schema)
table = client.create_table(table, exists_ok=True)

# Load the dataframe into BigQuery
job_config = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition='WRITE_TRUNCATE'  # replace data on each run
)

job = client.load_table_from_dataframe(df, full_table_id, job_config=job_config)
job.result()  # wait for job to complete

# Confirm
table = client.get_table(full_table_id)
print(f"Loaded {table.num_rows} rows to {full_table_id}")

Loaded 7 rows to crafty-circlet-491120-t3.dbt_wfiore.raw_tampa_weather
